# Crypto Strategy Backtester: Research Walkthrough

This notebook is a thin research interface over the tested `crypto_backtester` package. Core data, strategy, cost, backtest, and metric logic lives under `src/`, where it is covered by automated tests.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from crypto_backtester.config import load_config

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

## 1. Inspect the fixed experiment

The public YAML file is the complete experiment contract: assets, time range, fixed signal windows, exposure limits, cost scenarios, evaluation periods, and output location.

In [ ]:
config_path = PROJECT_ROOT / "configs" / "full_sample_baseline.yaml"
config = load_config(config_path)
config

## 2. Reproduce (optional)

The command below downloads or reuses the ignored local cache, validates the UTC time grid, runs both strategies under three cost scenarios, and atomically publishes artifacts. Uncomment when network access is available.

In [ ]:
# from crypto_backtester.pipeline import run_experiment
# output_dir = run_experiment(config, allow_download=True)
# output_dir

## 3. Read committed baseline results

The failed strategy remains visible. Return is always shown with risk and cost information.

In [ ]:
metrics_path = (
    PROJECT_ROOT / "artifacts" / "full_sample_baseline" / "metrics" / "performance_summary.csv"
)
metrics = pd.read_csv(metrics_path)
baseline = metrics.loc[
    metrics["cost_scenario"].eq("baseline"),
    [
        "strategy",
        "net_return",
        "sharpe_net",
        "sortino_net",
        "max_drawdown_net",
        "total_turnover_usdt",
        "total_cost_usdt",
    ],
]
baseline.style.format(
    {
        "net_return": "{:.2%}",
        "sharpe_net": "{:.3f}",
        "sortino_net": "{:.3f}",
        "max_drawdown_net": "{:.2%}",
        "total_turnover_usdt": "{:,.0f}",
        "total_cost_usdt": "{:,.0f}",
    }
)

In [ ]:
figure_path = (
    PROJECT_ROOT
    / "artifacts"
    / "full_sample_baseline"
    / "figures"
    / "full_sample_equity_curves.png"
)
display(Image(filename=str(figure_path), width=950))

## 4. Compare development and holdout periods

The fixed trend rule performs strongly in 2024 and loses almost all capital in the illustrative 2025 holdout. This instability is the central research finding.

In [ ]:
holdout_path = (
    PROJECT_ROOT / "artifacts" / "holdout_evaluation" / "metrics" / "performance_summary.csv"
)
holdout = pd.read_csv(holdout_path)
holdout.loc[
    holdout["cost_scenario"].eq("baseline"),
    ["period", "strategy", "net_return", "sharpe_net", "max_drawdown_net"],
].style.format({"net_return": "{:.2%}", "sharpe_net": "{:.3f}", "max_drawdown_net": "{:.2%}"})

## Interpretation

The full-sample trend profit is not sufficient evidence of robustness. Its low Sharpe ratio, severe drawdown, high turnover, cost sensitivity, and holdout failure argue for volatility-aware sizing, stronger execution modeling, walk-forward validation, and forward paper trading before any deployment consideration.